In [ ]:
# @title Google Colab setup (run this cell first, then restart runtime)  {display-mode: "form"}
import importlib
import os
import subprocess
import sys

# Clone the repo if not already present
if not os.path.isdir("/content/dl4bi"):
    subprocess.check_call([
        "git", "clone", "--branch", "claude/deeprv-nonnegative-tutorial-QQCvh",
        "https://github.com/flaxter/dl4bi.git", "/content/dl4bi"
    ])

# Add the package root to sys.path.
repo_root = "/content/dl4bi"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Install missing packages.
# Pin jax to avoid Colab's pre-installed version being downgraded by jraph/other deps.
packages = [
    "jax[cuda12]>=0.4.1",   # keep JAX version compatible with Colab GPU runtime
    "flax>=0.12",
    "hydra-core>=1.3",
    "numpyro",
    "jraph",
    "einops",
    "orbax-checkpoint",
    "scoringrules",
    "git+https://github.com/MLGlobalHealth/sps.git",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
importlib.invalidate_caches()

print("Setup complete. Please restart the Colab runtime now (Runtime → Restart runtime).")
print("After restarting, run all remaining cells.")
print(f"sys.path[0]: {sys.path[0]}")

# Matérn GP Prior for Effective Population Size with DeepRV

## Overview

The **Skygrid** coalescent model (Gill et al., 2013) infers the effective population
size $N_e(t)$ trajectory from a dated phylogeny. The key ingredient is a prior over
the $M$ log-population sizes $\boldsymbol{\gamma} = (\gamma_1, \ldots, \gamma_M)$
that enforces temporal smoothness.

### Standard approach: GMRF

The classic choice is a **first-order Gaussian Markov Random Field** (GMRF):

$$\boldsymbol{\gamma} \mid \tau \;\sim\; \mathcal{N}\!\left(\mathbf{0},\, \tau^{-1} R^{-1}\right)$$

where $R$ is a tridiagonal first-difference precision matrix. The GMRF is
*non-stationary* (intrinsic) — it penalises changes between adjacent bins but has
no marginal variance or length-scale. The tridiagonal structure makes Cholesky $O(M)$.

### This tutorial: Matérn 3/2 GP

We replace the GMRF with a **stationary Matérn 3/2 GP** over the time grid:

$$k(t_i, t_j) = \sigma^2\!\left(1 + \frac{\sqrt{3}\,|t_i - t_j|}{\ell}\right)
    \exp\!\left(-\frac{\sqrt{3}\,|t_i - t_j|}{\ell}\right)$$

This introduces two interpretable hyperparameters — marginal variance $\sigma^2$
and length-scale $\ell$ — that the data can inform. The covariance matrix $K(\sigma^2, \ell)$
is **dense** $M \times M$, so the Cholesky at every MCMC step costs $O(M^3)$.

### DeepRV to the rescue

**DeepRV** (Navott et al., arXiv 2503.21473) learns a neural surrogate

$$\hat{\mathbf{f}} = \mathrm{DeepRV}(\mathbf{z},\, \sigma^2,\, \ell)
    \approx L(\sigma^2, \ell)\,\mathbf{z}$$

trained once before MCMC. During inference the expensive Cholesky is replaced by
a cheap neural forward pass, enabling joint MCMC over $(\mathbf{z}, \sigma^2, \ell)$.

We demonstrate on the **HCV Egypt** dataset: 63 hepatitis-C sequences sampled in
Egypt in 1993, with 62 coalescent heights from a fixed MCC tree.

### Tutorial outline
1. Imports & constants
2. Matérn 3/2 kernel and training data
3. Train the DeepRV surrogate
4. Bayesian inference via NUTS (GMRF baseline vs Matérn GP + DeepRV)
5. Visualise and compare $N_e(t)$ posteriors

## 1. Imports & constants

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import io, contextlib
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import optax
import wandb
from jax import jit, random
from matplotlib.lines import Line2D
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

from dl4bi.coalescent.skygrid import gmrf_log_prob, skygrid_coalescent_log_prob
from dl4bi.coalescent.hcv_egypt import HEIGHTS, SAMPLING_TIMES
from dl4bi.core.model_output import VAEOutput
from dl4bi.core.train import Callback, cosine_annealing_lr, train
from dl4bi.vae import gMLPDeepRV
from dl4bi.vae.train_utils import deep_rv_train_step, generate_surrogate_decoder

wandb.init(mode="disabled")

In [ ]:
# ── Grid parameters ────────────────────────────────────────────────────────────
M       = 20                                    # number of grid intervals
CUTOFF  = float(HEIGHTS[-1]) * 1.1              # ~ 305.8 years before 1993

grid_full   = jnp.linspace(0, CUTOFF, M + 1)   # 21 boundary points
grid        = grid_full[1:-1]                   # (19,) interior change-points
grid_times  = ((grid_full[:-1] + grid_full[1:]) / 2)[:, None]  # (M, 1) midpoints
times_flat  = grid_times[:, 0]                  # (M,) for kernel computation

# ── DeepRV "locations" ────────────────────────────────────────────────────────
# gMLPDeepRV is a spatial model; each grid interval is a "location" on the time axis.
s = grid_times   # shape (M, 1)

SEED = 42
SAMPLING_YEAR = 1993.0

print(f"M = {M}  grid intervals")
print(f"CUTOFF = {CUTOFF:.1f} years before present")
print(f"Interval width = {CUTOFF / M:.1f} years")
print(f"s.shape = {s.shape}")
print(f"Coalescent events: {len(HEIGHTS)}")

## 2. Matérn 3/2 kernel

The Matérn 3/2 covariance between grid points $t_i$ and $t_j$ is

$$K_{ij} = \sigma^2 \left(1 + \frac{\sqrt{3}\,|t_i - t_j|}{\ell}\right)
           \exp\!\left(-\frac{\sqrt{3}\,|t_i - t_j|}{\ell}\right)$$

This gives a once-differentiable GP — smoother than Matérn 1/2 (Ornstein--Uhlenbeck)
but rougher than the squared-exponential. The hyperparameters are:

* $\sigma^2$ — marginal variance of $\log N_e(t)$
* $\ell$ — length-scale: how quickly correlations decay over time

DeepRV learns the map $\mathbf{z} \to L(\sigma^2, \ell)\,\mathbf{z}$ simultaneously
across the distribution of $(\sigma^2, \ell)$ specified by a training prior, so MCMC
can vary these hyperparameters freely without recomputing the Cholesky.

In [ ]:
@jit
def matern32_kernel(times, log_sigma2, log_ell, jitter=1e-5):
    """Matérn 3/2 covariance matrix on a 1-D time grid.

    Parameters
    ----------
    times      : (M,) array – grid midpoint times
    log_sigma2 : scalar     – log marginal variance
    log_ell    : scalar     – log length-scale

    Returns
    -------
    K : (M, M) positive-definite covariance matrix
    """
    sigma2 = jnp.exp(log_sigma2)
    ell    = jnp.exp(log_ell)
    d = jnp.abs(times[:, None] - times[None, :])   # (M, M)
    r = jnp.sqrt(3.0) * d / ell
    K = sigma2 * (1.0 + r) * jnp.exp(-r)
    return K + jitter * jnp.eye(times.shape[0])


@jit
def gp_cholesky(times, log_sigma2, log_ell):
    """Lower Cholesky factor of K(sigma2, ell)."""
    return jnp.linalg.cholesky(matern32_kernel(times, log_sigma2, log_ell))

## 3. Training data for DeepRV

Each training batch samples a fresh $(\sigma^2, \ell)$ pair from a broad prior,
computes the exact Cholesky $L$, and generates paired $(\mathbf{z},\, \mathbf{f} = L\mathbf{z})$
examples. After training, DeepRV can reproduce $L(\sigma^2, \ell)\,\mathbf{z}$ for
any $(\sigma^2, \ell)$ in the prior support — including values never seen during training.

In [ ]:
# Training priors for kernel hyperparameters
LOG_SIGMA2_MEAN = 0.0;   LOG_SIGMA2_STD = 2.0     # sigma2 in ~[0.02, 55]
LOG_ELL_MEAN    = float(jnp.log(CUTOFF / 4));  LOG_ELL_STD = 1.0  # ell centered ~ 76 yr


def gen_train_dataloader(s, batch_size=64):
    """Infinite generator of (z, f, conditionals, s) training batches."""
    def dataloader(rng):
        while True:
            rng, rng_s2, rng_ell, rng_z = random.split(rng, 4)
            log_sigma2 = dist.Normal(LOG_SIGMA2_MEAN, LOG_SIGMA2_STD).sample(rng_s2)
            log_ell    = dist.Normal(LOG_ELL_MEAN,    LOG_ELL_STD).sample(rng_ell)

            L = gp_cholesky(times_flat, log_sigma2, log_ell)
            z = dist.Normal().sample(rng_z, (batch_size, M))
            f = jnp.einsum("ij,bj->bi", L, z)      # (batch, M)

            yield {
                "s":            s,
                "z":            z,
                "conditionals": jnp.array([log_sigma2, log_ell]),
                "f":            f,
            }
    return dataloader

print(f"Training prior: log sigma2 ~ N({LOG_SIGMA2_MEAN}, {LOG_SIGMA2_STD})")
print(f"Training prior: log ell    ~ N({LOG_ELL_MEAN:.3f}, {LOG_ELL_STD})")

## 4. Train the DeepRV surrogate

`gMLPDeepRV` is a gated-MLP decoder: given latent noise $\mathbf{z} \in \mathbb{R}^M$,
locations $\mathbf{s}$, and conditionals $(\log\sigma^2, \log\ell)$, it outputs
$\hat{\mathbf{f}} \in \mathbb{R}^M$. Training minimises the MSE between $\hat{\mathbf{f}}$
and the exact $L(\sigma^2, \ell)\,\mathbf{z}$.

In [ ]:
N_STEPS        = 8_000
SNAPSHOT_STEPS = {500, 2_000, 5_000, 8_000}

rng = random.key(SEED)
rng_train, rng_infer_gmrf, rng_infer_drv, rng_vis = random.split(rng, 4)

# ── Fixed held-out batch for monitoring ────────────────────────────────────────
rng_cb      = random.fold_in(rng_vis, 101)
Z_CB        = dist.Normal().sample(rng_cb, (50, M))
LS2_CB      = jnp.log(jnp.array(2.0))          # sigma2 = 2
LELL_CB     = jnp.log(jnp.array(CUTOFF / 4))   # ell = cutoff/4
L_CB        = gp_cholesky(times_flat, LS2_CB, LELL_CB)
F_CB_EXACT  = jnp.einsum("ij,bj->bi", L_CB, Z_CB)   # (50, M)

curve_steps, curve_rmse, snapshots = [], [], {}

nn_model = gMLPDeepRV(num_blks=3)

def record_progress(step, rng_step, state, batch, extra):
    f_hat = nn_model.apply(
        {"params": state.params, **state.kwargs},
        Z_CB, jnp.array([LS2_CB, LELL_CB]), s=s,
        rngs={"extra": rng_step},
    ).f_hat.squeeze(-1)   # (50, M)
    rmse = float(jnp.sqrt(jnp.mean((f_hat - F_CB_EXACT) ** 2)))
    curve_steps.append(step)
    curve_rmse.append(rmse)
    if step in SNAPSHOT_STEPS:
        snapshots[step] = (state.params, state.kwargs)

optimizer = optax.chain(
    optax.clip_by_global_norm(3.0),
    optax.adamw(cosine_annealing_lr(N_STEPS, 1e-3), weight_decay=1e-2),
)
loader = gen_train_dataloader(s)

with contextlib.redirect_stdout(io.StringIO()):
    state = train(
        rng_train, nn_model, optimizer, deep_rv_train_step,
        N_STEPS, loader,
        callbacks=[Callback(fn=record_progress, interval=200)],
    )

surrogate_decoder = generate_surrogate_decoder(state, nn_model)
print("DeepRV training complete.")

### Training curve and trajectory snapshots

The callback recorded RMSE on a fixed held-out batch (50 samples, $\sigma^2 = 2$,
$\ell = \text{cutoff}/4$) every 200 steps. The bottom row shows sample GP
trajectories: exact $L\mathbf{z}$ (blue) vs DeepRV $\hat{\mathbf{f}}$ (red dashed).

In [ ]:
snap_steps_sorted = sorted(snapshots.keys())
n_traj = 8

fig = plt.figure(figsize=(18, 9))
gs  = fig.add_gridspec(2, len(snap_steps_sorted), hspace=0.45, wspace=0.28)

# ── top row: training curve ────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[0, :])
ax_c.plot(curve_steps, curve_rmse, color="steelblue", lw=1.5)
for step in snap_steps_sorted:
    ax_c.axvline(step, color="crimson", ls="--", lw=1, alpha=0.7)
    ax_c.text(step, max(curve_rmse) * 0.97, f" {step:,}", color="crimson",
              fontsize=8, va="top")
ax_c.set_xlabel("Training step")
ax_c.set_ylabel(r"RMSE  (held-out, $\sigma^2$=2, $\ell$=cutoff/4)")
ax_c.set_title("DeepRV training curve")
ax_c.set_yscale("log")
ax_c.grid(True, which="both", alpha=0.3)

# ── bottom row: trajectory snapshots ──────────────────────────────────────────
t_axis = np.array(times_flat)

for col, step in enumerate(snap_steps_sorted):
    ax = fig.add_subplot(gs[1, col])
    params, kwargs = snapshots[step]

    f_hat_snap = nn_model.apply(
        {"params": params, **kwargs},
        Z_CB[:n_traj], jnp.array([LS2_CB, LELL_CB]), s=s,
        rngs={"extra": random.fold_in(rng_vis, step)},
    ).f_hat.squeeze(-1)   # (n_traj, M)

    for i in range(n_traj):
        ax.plot(t_axis, F_CB_EXACT[i], color="steelblue", alpha=0.5, lw=1.2)
        ax.plot(t_axis, np.array(f_hat_snap[i]), color="tomato",
                alpha=0.5, lw=1.2, ls="--")

    ax.set_title(f"Step {step:,}")
    ax.set_xlabel("Years before present")
    if col == 0:
        ax.set_ylabel("log Ne")
    if col == len(snap_steps_sorted) - 1:
        ax.legend(
            [Line2D([0], [0], color="steelblue"),
             Line2D([0], [0], color="tomato", ls="--")],
            ["Exact L z", "DeepRV"], fontsize=8, loc="upper right",
        )

plt.suptitle(r"DeepRV approximation quality across training ($\sigma^2$=2, $\ell$=cutoff/4)",
             fontsize=12, y=1.02)
plt.show()

### Sample quality across $(\sigma^2, \ell)$ values

DeepRV was trained jointly over a distribution of kernel hyperparameters.
Below we verify it generalises: each panel tests a different $(\sigma^2, \ell)$
pair not seen verbatim during training.

In [ ]:
sigma2_ell_grid = [
    (0.5, CUTOFF / 8),
    (1.0, CUTOFF / 4),
    (2.0, CUTOFF / 2),
    (4.0, CUTOFF),
]

n_check   = 300
rng_check = random.fold_in(rng_vis, 99)
z_check   = dist.Normal().sample(rng_check, (n_check, M))
idx       = M // 2   # grid index to scatter

fig, axes = plt.subplots(1, len(sigma2_ell_grid), figsize=(16, 4), sharey=True)
for ax, (s2, ell) in zip(axes, sigma2_ell_grid):
    ls2  = jnp.log(jnp.array(s2))
    lell = jnp.log(jnp.array(ell))
    L_ref   = gp_cholesky(times_flat, ls2, lell)
    f_exact = jnp.einsum("ij,bj->bi", L_ref, z_check)   # (n_check, M)
    f_hat   = surrogate_decoder(z_check, jnp.array([ls2, lell]), s=s).squeeze(-1)

    ax.scatter(f_exact[:, idx], f_hat[:, idx], alpha=0.3, s=8, color="steelblue")
    lim = max(abs(f_exact[:, idx]).max(), abs(f_hat[:, idx]).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
    rmse = float(jnp.sqrt(jnp.mean((f_hat[:, idx] - f_exact[:, idx]) ** 2)))
    ax.text(0.05, 0.92, f"RMSE={rmse:.4f}", transform=ax.transAxes, fontsize=8)
    ax.set_xlabel(f"Exact  f[{idx}]")
    if ax is axes[0]:
        ax.set_ylabel("DeepRV  f_hat")
    ax.set_title(f"$\\sigma^2$={s2}, $\\ell$={ell:.0f}")
    ax.set_aspect("equal")

plt.suptitle(f"DeepRV vs exact at grid index {idx} (t = {float(t_axis[idx]):.0f} yr BP)",
             fontsize=11)
plt.tight_layout()
plt.show()

## 5. Bayesian inference on HCV Egypt

The HCV Egypt dataset has 63 hepatitis-C E1 sequences sampled in Egypt in 1993.
We compare two priors for $\boldsymbol{\gamma} = \log N_e(t)$:

| Model | Prior on $\boldsymbol{\gamma}$ |
|---|---|
| **GMRF** (Gill et al. 2013) | First-order RW; precision $\tau \sim \text{Gamma}(0.001, 0.001)$ |
| **Matérn GP + DeepRV** | Matérn 3/2; $\sigma^2, \ell$ inferred jointly via NUTS |

Both share the same piecewise-constant coalescent likelihood.

In [ ]:
def gmrf_model(node_heights, sampling_times, grid):
    """Standard GMRF Skygrid model (baseline)."""
    precision = numpyro.sample("precision", dist.Gamma(0.001, 0.001))
    log_thetas = numpyro.sample(
        "log_thetas", dist.Normal(0.0, 10.0).expand([M]).to_event(1))
    numpyro.factor("gmrf", gmrf_log_prob(log_thetas, precision))
    numpyro.factor("coalescent",
        skygrid_coalescent_log_prob(log_thetas, grid, node_heights, sampling_times))


def matern_deeprv_model(surrogate_decoder, node_heights, sampling_times, grid):
    """Matérn GP prior on log Ne(t), reparameterised via DeepRV."""
    # Kernel hyperparameters
    log_sigma2 = numpyro.sample("log_sigma2",
                                dist.Normal(LOG_SIGMA2_MEAN, LOG_SIGMA2_STD))
    log_ell    = numpyro.sample("log_ell",
                                dist.Normal(LOG_ELL_MEAN, LOG_ELL_STD))

    # Latent Gaussian noise
    z = numpyro.sample("z", dist.Normal(0.0, 1.0).expand([M]).to_event(1))

    # DeepRV: z + kernel hyperparams → log_thetas
    conditionals = jnp.array([log_sigma2, log_ell])
    log_thetas = numpyro.deterministic(
        "log_thetas",
        surrogate_decoder(z[None], conditionals, s=s)[0, :, 0],
    )

    # Coalescent likelihood
    numpyro.factor("coalescent",
        skygrid_coalescent_log_prob(log_thetas, grid, node_heights, sampling_times))

In [ ]:
def run_nuts(rng_key, model, label, **model_kwargs):
    kernel = NUTS(model, max_tree_depth=10)
    mcmc = MCMC(kernel, num_warmup=500, num_samples=1000,
                num_chains=1, progress_bar=True)
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    mcmc.run(rng_key, **model_kwargs)
    mcmc.print_summary(exclude_deterministic=True)
    return mcmc


# ── Run GMRF baseline ─────────────────────────────────────────────────────────
mcmc_gmrf = run_nuts(
    rng_infer_gmrf, gmrf_model, "GMRF (baseline)",
    node_heights=HEIGHTS, sampling_times=SAMPLING_TIMES, grid=grid,
)

In [ ]:
# ── Run Matérn GP + DeepRV ─────────────────────────────────────────────────────
mcmc_drv = run_nuts(
    rng_infer_drv, matern_deeprv_model, "Matérn 3/2 GP + DeepRV",
    surrogate_decoder=surrogate_decoder,
    node_heights=HEIGHTS, sampling_times=SAMPLING_TIMES, grid=grid,
)

## 6. Results: posterior $N_e(t)$ trajectories

We compare the posterior median and 95% credible interval for $N_e(t)$ under
both priors. Calendar years on the x-axis (1993 minus years-before-present).

In [ ]:
def plot_ne(ax, mcmc, label, color):
    """Plot posterior median + 95% CI of Ne(t)."""
    log_thetas = mcmc.get_samples()["log_thetas"]   # (S, M)
    ne = np.exp(np.array(log_thetas))
    median = np.median(ne, axis=0)
    lo     = np.percentile(ne, 2.5, axis=0)
    hi     = np.percentile(ne, 97.5, axis=0)
    cal    = SAMPLING_YEAR - t_axis   # calendar years

    ax.fill_between(cal, lo, hi, alpha=0.25, color=color)
    ax.plot(cal, median, color=color, lw=2, label=label)


fig, ax = plt.subplots(figsize=(11, 5))
plot_ne(ax, mcmc_gmrf, "GMRF (baseline)", color="steelblue")
plot_ne(ax, mcmc_drv,  "Matérn 3/2 GP + DeepRV", color="tomato")

ax.set_yscale("log")
ax.invert_xaxis()
ax.set_xlabel("Calendar year")
ax.set_ylabel("Effective population size  $N_e(t)$")
ax.set_title("HCV Egypt: posterior $N_e(t)$  --  GMRF vs Matérn GP prior")
ax.legend(fontsize=11)
ax.grid(True, which="both", alpha=0.2)
fig.tight_layout()
plt.show()

### Inferred Matérn GP hyperparameters

The Matérn model yields a posterior over $\sigma^2$ and $\ell$, telling us about the
amplitude and time-scale of $N_e(t)$ variation as informed by the data.

In [ ]:
samples_drv = mcmc_drv.get_samples()
sigma2_post = np.exp(np.array(samples_drv["log_sigma2"]))
ell_post    = np.exp(np.array(samples_drv["log_ell"]))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(sigma2_post, bins=30, color="tomato", alpha=0.7, edgecolor="white")
axes[0].set_xlabel(r"$\sigma^2$  (GP marginal variance)")
axes[0].set_ylabel("Count")
axes[0].set_title(r"Posterior of $\sigma^2$")

axes[1].hist(ell_post, bins=30, color="tomato", alpha=0.7, edgecolor="white")
axes[1].axvline(CUTOFF, color="k", ls="--", lw=1.2,
                label=f"cutoff = {CUTOFF:.0f} yr")
axes[1].set_xlabel(r"$\ell$  (length-scale, years)")
axes[1].set_title(r"Posterior of $\ell$")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"sigma2: median = {np.median(sigma2_post):.2f}, "
      f"95% CI = [{np.percentile(sigma2_post, 2.5):.2f}, "
      f"{np.percentile(sigma2_post, 97.5):.2f}]")
print(f"ell:    median = {np.median(ell_post):.1f} yr, "
      f"95% CI = [{np.percentile(ell_post, 2.5):.1f}, "
      f"{np.percentile(ell_post, 97.5):.1f}] yr")

## Summary

| | GMRF (baseline) | Matérn 3/2 GP + DeepRV |
|---|---|---|
| **Prior** | Intrinsic first-order RW | Stationary Matérn 3/2 |
| **Hyperparameters** | $\tau$ (precision) | $\sigma^2$ (variance), $\ell$ (length-scale) |
| **Length-scale** | None -- all adjacent changes penalised equally | Explicit $\ell$ inferred from data |
| **Cholesky cost per MCMC step** | $O(M)$ tridiagonal | $O(1)$ neural forward pass |
| **Training cost** | None | $O(N_{\text{steps}})$ one-time |

**Key takeaways**:

1. **DeepRV makes stationary GP priors practical** for Skygrid coalescent inference.
   The dense $M \times M$ Cholesky is computed during training only; at MCMC time, the
   surrogate replaces it with a cheap forward pass.

2. **The Matérn GP prior has interpretable hyperparameters**: the posterior over $\ell$
   tells us the characteristic time-scale of $N_e(t)$ variation, while $\sigma^2$
   controls the amplitude. The GMRF offers no such decomposition.

3. **Scaling**: For $M = 20$ both methods are fast. The DeepRV advantage grows with $M$:
   at $M = 500$, the dense Cholesky costs $O(M^3) \approx 1.25 \times 10^8$ operations
   per MCMC step, while DeepRV remains $O(1)$ after training.

4. **This is novel**: no prior work has applied a stationary GP kernel (Matérn, RBF)
   directly to the Skygrid time grid. Suchard's group uses GMRF throughout (including
   Monti et al. 2026 which uses a GP over *covariates* but keeps GMRF for time).
   Palacios & Minin (2013) used a non-stationary integrated Brownian motion GP.